In [1]:
import torch 
import pandas as pd 
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from torch.utils.data import Dataset
from torch.utils.data import random_split 
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from torch.optim import Adam
from tqdm import tqdm
from itertools import islice
from Early_stopping_class import EarlyStopping
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch.nn as nn
import time
import torch.nn.functional as F
import re



In [2]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
local_model_path = "models/Qwen/Qwen3-1.7B" # Path to the downloaded model directory
device = torch.device("cuda:0")


# Load model in 4-bit to save memory (important for keyboard research)
Qwen_model = AutoModelForCausalLM.from_pretrained(
    local_model_path, 
    #device_map={ " " : 1},  # Automatically distribute layers across available devices
    torch_dtype=torch.float16,
    #load_in_4bit=True,
    local_files_only=True,
    trust_remote_code=True,
).to(device)

Qwen_model_router = AutoModelForCausalLM.from_pretrained(
    local_model_path, 
    #device_map={ " " : 1},  # Automatically distribute layers across available devices
    torch_dtype=torch.float16,
    #load_in_4bit=True,
    local_files_only=True,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    local_model_path, 
    local_files_only=True,
    trust_remote_code=True)

Qwen_model = PeftModel.from_pretrained(
    Qwen_model,
    "./best_adapter_enfr",
    adapter_name="fr_adapter",
    is_trainable=False
)

Qwen_model.load_adapter(
    "./zh_en_adapter_best",
    adapter_name="zh_adapter"
)

#Qwen_model.set_adapter("fr_adapter")  # or zh, just to initialize
#Qwen_model.disable_adapter_layers()   # THIS is the correct API

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


<All keys matched successfully>

In [3]:
print(f"Active Adapter Name: {Qwen_model.active_adapter}")

Active Adapter Name: fr_adapter


In [16]:
class MultilingualKeyboardEngine:
    def __init__(self, base_model, base_model_router ,tokenizer, router, device):
        self.model = base_model
        self.model_router = base_model_router
        self.tokenizer = tokenizer
        self.router = router.to(device).float()
        self.device = device
        self.mapping = {0: "base", 1: "fr_adapter", 2: "zh_adapter"}

    def get_next_word(self, text):
        if not text.strip():
            return "", "None", 0.0
        # --- 1. ROUTING ---
        # Get embedding using the "Last Token" method we perfected
        self.router.eval()
        self.model_router.eval()
        self.model.eval()
        inputs = self.tokenizer(text, return_tensors="pt", padding=True , truncation=True,).to(self.device)
        with torch.no_grad():
            outputs = self.model_router(**inputs, output_hidden_states=True)
            last_layer = outputs.hidden_states[-1]
            embedding = last_layer[:, -1, :].float()  
            logits = self.router(embedding)
            probs = F.softmax(logits, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_idx].item()

        # --- 2. ADAPTER SWITCHING ---
        if pred_idx == 2: # Router picked Chinese
            if not re.search(r'[\u4e00-\u9fff]', text): # No Chinese characters found
                pred_idx = 0 # Force back to Base (English)
        
        target_adapter = self.mapping[pred_idx]

        
        if target_adapter == "base":
            #self.model.disable_adapter()
            self.model.disable_adapter_layers()
        else:
            self.model.set_adapter(target_adapter)
            self.model.enable_adapter_layers()
            
        # --- 3. GENERATION ---
        # Now generate the next token using the active specialist
        with torch.no_grad():
            gen_outputs = self.model.generate(
                **inputs, 
                max_new_tokens=5, 
                do_sample=False # For a keyboard, we usually want the most likely word
            )

        input_length = inputs.input_ids.shape[1]
        new_tokens = gen_outputs[0][input_length:]
        prediction = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return prediction.strip(), target_adapter, confidence

In [5]:
class Router(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )
    
    def forward(self,x):
        return self.mlp(x)
    

In [6]:
torch.manual_seed(42)
router = Router(hidden_size=2048).to(device)

router.load_state_dict(torch.load(f="./router_best.pth"))

router.to(device)

Router(
  (mlp): Sequential(
    (0): Linear(in_features=2048, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [28]:
import torch.nn.functional as F
import time

# 1. Instantiate (Ensure your router is loaded as 'router' and on the right device)
#router = router.to(device).float()

engine = MultilingualKeyboardEngine(
    base_model=Qwen_model,
    base_model_router=Qwen_model_router, 
    tokenizer=tokenizer, 
    router=router, 
    device=device
)

test_sentences = [
    "I'm currently working on my master's thesis in NLP.", # English
    "Je voudrais un croissant et un café, s'il vous plaît.", # French
    "你好，我正在学习人工智能技术。",                         # Chinese
    "Let's meet at the café for a quick rendezvous.",      # Mixed (EN/FR)
    "I really like this 你好 style of greeting." , 
    "Bonjour , je m'appelle Nour et j'habite à Paris."         # Mixed (EN/ZH)
]

# --- 3. Execute Tests ---
print(f"{'Text':<50} | {'Language':<10} | {'Confidence'} | {"Prediction"}")
print("-" * 100)

for text in test_sentences:
    word, lang, conf = engine.get_next_word(text)
    print(f"{text[:60]:<50} | {lang:<10} | {conf*100:.2f}% | {word}")
   



Text                                               | Language   | Confidence | Prediction
----------------------------------------------------------------------------------------------------
I'm currently working on my master's thesis in NLP. | base       | 83.21% | I need to create a
Je voudrais un croissant et un café, s'il vous plaît. | fr_adapter | 99.93% | I'm hungry, you
你好，我正在学习人工智能技术。                                    | zh_adapter | 99.98% | I'm not going
Let's meet at the café for a quick rendezvous.     | base       | 99.94% | I'm going to bring
I really like this 你好 style of greeting.           | zh_adapter | 99.95% | And I think that
Bonjour , je m'appelle Nour et j'habite à Paris.   | fr_adapter | 100.00% | I like going to the


In [8]:
import torch.nn.functional as F
import time
import re # Make sure regex is imported

# --- 1. Setup (Assuming engine is already defined as 'engine') ---

engine = MultilingualKeyboardEngine(
    base_model=Qwen_model,
    base_model_router=Qwen_model_router, 
    tokenizer=tokenizer, 
    router=router, 
    device=device
)
test_sentences = [
    # pure examples
    "I'm currently working on my master's thesis in NLP",
    "Je voudrais un croissant et un café, s'il vous pla", 
    "你好，我正在学习人工智能技",                         
    # mixed/complex examples
    "Let's meet at the café for a quick rendezvous",      
    "I really like this 你好 style of greeting", 
    "Bonjour, je m'appelle Nour et j'habite à Paris",
    # short/bias examples (The 'Crucial Tests')
    "I love",
    "Hey bro, how are you?",
    "i hope you are doing well"
]

# --- 2. Defining a Better Printer Function ---

def get_formatted_result(text):
    """
    Simulates the actual get_next_word, including the heuristic check,
    and formats the output for the report.
    """
    # 1. Routing step (assuming your engine's internal call)
    # _, pred_idx, conf = engine.route_text(text)
    
    # Simulate your current 'get_next_word' behavior:
    full_prediction, raw_lang, conf = engine.get_next_word(text)
    
    # -------------------------------------------------------------
    # THIS IS THE LOGIC YOU ADDED (The 'Cheating'/Optimization fix)
    # We must explicitly show it in the results.
    # -------------------------------------------------------------
    if raw_lang == "zh_adapter" and not re.search(r'[\u4e00-\u9fff]', text):
        final_lang = "base (Gate)" # Clarify that the gate saved it.
    else:
        final_lang = raw_lang

    # Clean the predicted text for the table (removes brackets, full prompt)
    clean_text = full_prediction[len(text):].strip()
    # Remove brackets/quotes that appear in your raw results
    clean_text = clean_text.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    # Grab the first two words (good for table presentation)
    final_word_suggestion = " ".join(clean_text.split()[:2])
    
    return {
        "text": text,
        "final_lang": final_lang,
        "conf": conf,
        "raw_suggestion": full_prediction, # Keep the full string for context
        "ui_suggestion": final_word_suggestion
    }

# --- 3. Run and Print the Table ---

results = [get_formatted_result(t) for t in test_sentences]

# Print Table Header
header = f"{'Input Text (Last Words)':<50} | {'Adapter'::<14} | {'Conf.':<7} | {'Generated Completion':<30} | {'Extracted UI Suggestion'}"
separator = "-" * (len(header) + 20)
print(separator)
print(header)
print(separator)

# Print Rows
for r in results:
    # Truncate text for table look
    short_input = r['text'][:47] + "..." if len(r['text']) > 47 else r['text']
    
    print(f"{short_input:<50} | {r['final_lang']:<14} | {r['conf']*100:.1f}% | {r['raw_suggestion'][:27]+'...':<30} | {r['ui_suggestion']}")

print(separator)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AttributeError: 'list' object has no attribute 'strip'

In [35]:
# 1. Instantiate the Engine
engine = MultilingualKeyboardEngine(
    base_model=Qwen_model, 
    base_model_router=Qwen_model_router,
    tokenizer=tokenizer, 
    router=router, 
    device=device
)

# 2. Define a helper for clean output
def test_keyboard(input_text):
    start = time.time()
    word, lang, conf = engine.get_next_word(input_text)
    duration = (time.time() - start) * 1000
    
    print(f"User Typed: '{input_text}'")
    print(f"Suggested : '{word}'")
    print(f"Detected  : {lang} ({conf:.2%})")
    print(f"Latency   : {duration:.2f}ms")
    print("-" * 50)

In [40]:
test_keyboard("Let's meet at the café for a quick rendezvous.")

User Typed: 'Let's meet at the café for a quick rendezvous.'
Suggested : '["Let's meet at the café for a quick rendezvous. I'm going to bring"]'
Detected  : base (99.94%)
Latency   : 214.40ms
--------------------------------------------------


In [38]:
def check_indices(engine):
    test_data = {
        "Hello my friend": "English",
        "Bonjour mon ami": "French",
        "你好我的朋友": "Chinese"
    }
    
    engine.model.disable_adapter()
    for text, lang in test_data.items():
        inputs = engine.tokenizer(text, return_tensors="pt").to(engine.device)
        with torch.no_grad():
            out = engine.model(**inputs, output_hidden_states=True)
            emb = out.hidden_states[-1][:, -1, :].float()
            logits = engine.router(emb)
            idx = lang
            print(f"Text: {text} | Expected: {lang} | Router Predicted Index: {idx}")

check_indices(engine)

Text: Hello my friend | Expected: English | Router Predicted Index: English
Text: Bonjour mon ami | Expected: French | Router Predicted Index: French
Text: 你好我的朋友 | Expected: Chinese | Router Predicted Index: Chinese


In [41]:
import time

def benchmark_system(text):
    start_total = time.time()
    
    # 1. Routing Time
    start_route = time.time()
    word, adapter, conf = engine.get_next_word(text)
    route_time = (time.time() - start_route) * 1000 # convert to ms
    
    total_time = (time.time() - start_total) * 1000
    
    return {
        "text_len": len(text),
        "adapter": adapter,
        "route_ms": round(route_time, 2),
        "total_ms": round(total_time, 2)
    }

# Test cases
test_inputs = ["Hello", "Bonjour, je", "你好，我", "This is a long sentence to test processing"]
results = [benchmark_system(t) for t in test_inputs]

import pandas as pd
print(pd.DataFrame(results))

   text_len     adapter  route_ms  total_ms
0         5        base    215.39    215.39
1        11  fr_adapter    236.66    236.66
2         4  zh_adapter    236.54    236.54
3        42        base    197.26    197.26
